In [9]:
import pandas as pd
import glob

if glob.glob("data/*.csv"):
    folder = "data"
else:
    folder = "tubes/data"

frames = []
for file in sorted(glob.glob(folder + "/*.csv")):
    frames.append(pd.read_csv(file))

df = pd.concat(frames, ignore_index=True)
df["datetime_from_utc"] = pd.to_datetime(df["datetime_from_utc"], utc=True)

print("Jumlah baris :", len(df))
print("Jumlah kolom :", len(df.columns))
print("Kolom        :", list(df.columns))

Jumlah baris : 6178
Jumlah kolom : 21
Kolom        : ['lokasi', 'waktu', 'pm1', 'pm25', 'um003', 'sensor_pm1', 'sensor_pm25', 'sensor_um003', 'location_id', 'sensor_id', 'parameter', 'units', 'value', 'datetime_from_utc', 'datetime_from_local', 'datetime_to_utc', 'datetime_to_local', 'min', 'max', 'avg', 'coverage_percent']


In [10]:
for lokasi in sorted(df["location_id"].unique()):
    data = df[df["location_id"] == lokasi]
    zona = data["datetime_from_local"].dropna().astype(str)

    print("=== LOKASI", lokasi, "===")
    print("Baris        :", len(data))
    print("Parameter    :", ", ".join(sorted(data["parameter"].unique())))
    print("Zona waktu   :", zona.iloc[0][-6:] if not zona.empty else "tidak tersedia")
    print("Mulai (UTC)  :", data["datetime_from_utc"].min())
    print("Selesai (UTC):", data["datetime_from_utc"].max())
    print("Duplikat     :", data.duplicated().sum())
    print("Data kosong  :", data.isna().sum().sum())
    print()

=== LOKASI nan ===
Baris        : 0
Parameter    : 
Zona waktu   : tidak tersedia
Mulai (UTC)  : NaT
Selesai (UTC): NaT
Duplikat     : 0
Data kosong  : 0

=== LOKASI 5586536.0 ===
Baris        : 651
Parameter    : pm1, pm25, um003
Zona waktu   : +07:00
Mulai (UTC)  : 2026-10-11 11:00:00+00:00
Selesai (UTC): 2026-10-20 11:00:00+00:00
Duplikat     : 0
Data kosong  : 5208

=== LOKASI 6494341.0 ===
Baris        : 2290
Parameter    : pm1, pm25, relativehumidity, temperature, um003
Zona waktu   : +08:00
Mulai (UTC)  : 2026-10-01 00:00:00+00:00
Selesai (UTC): 2026-10-20 11:00:00+00:00
Duplikat     : 0
Data kosong  : 18320

=== LOKASI 6551345.0 ===
Baris        : 2135
Parameter    : pm1, pm25, relativehumidity, temperature, um003
Zona waktu   : +07:00
Mulai (UTC)  : 2026-10-01 06:00:00+00:00
Selesai (UTC): 2026-10-20 11:00:00+00:00
Duplikat     : 0
Data kosong  : 17080



In [11]:
print("JUMLAH BARIS PER PARAMETER DAN LOKASI")
print(pd.crosstab(df["parameter"], df["location_id"]).to_string())
print()

print("STATISTIK NILAI")
print(df.groupby("parameter")["value"].describe().round(2).to_string())
print()

print("coverage_percent :", df["coverage_percent"].unique())
print("interval         :", (pd.to_datetime(df["datetime_to_utc"], utc=True) - df["datetime_from_utc"]).unique())
print("value != avg     :", (df["value"] != df["avg"]).sum(), "baris")

JUMLAH BARIS PER PARAMETER DAN LOKASI
location_id       5586536.0  6494341.0  6551345.0
parameter                                        
pm1                     217        458        427
pm25                    217        458        427
relativehumidity          0        458        427
temperature               0        458        427
um003                   217        458        427

STATISTIK NILAI
                   count     mean      std    min     25%     50%      75%      max
parameter                                                                          
pm1               1102.0    29.19    16.11    1.1    17.1    29.4    38.30     91.3
pm25              1102.0    49.98    26.61    4.6    29.7    47.8    65.15    188.0
relativehumidity   885.0    54.10     8.35   31.0    48.0    53.0    62.00     72.0
temperature        885.0    30.29     2.62   25.5    28.0    30.4    32.00     37.1
um003             1102.0  2964.40  2356.91  310.0  1260.0  2510.0  3797.50  17200.0

covera

In [12]:
daftar_lokasi = [6551345, 6494341, 5586536]
nama_lokasi = {6551345: "jakarta", 6494341: "bali", 5586536: "medan"}

# parameter yang ada di ketiga lokasi
isi_parameter = []
for lokasi in daftar_lokasi:
    isi_parameter.append(set(df[df["location_id"] == lokasi]["parameter"]))
parameter_umum = sorted(set.intersection(*isi_parameter))

print("Parameter dipakai:", parameter_umum)

def buat_tabel(lokasi):
    data = df[df["location_id"] == lokasi]
    tabel = data.pivot(index="datetime_from_utc", columns="parameter", values="value")
    tabel = tabel[parameter_umum]
    tabel.index.name = "waktu"
    return tabel

jakarta = buat_tabel(6551345)
bali = buat_tabel(6494341)
medan = buat_tabel(5586536)

Parameter dipakai: ['pm1', 'pm25', 'um003']


In [13]:
print("ID SENSOR")
for lokasi in daftar_lokasi:
    data = df[df["location_id"] == lokasi].drop_duplicates("parameter")
    print(nama_lokasi[lokasi])
    for _, baris in data.iterrows():
        print("  {:15} {}".format(baris["parameter"], baris["sensor_id"]))
print()

print("RINGKASAN TABEL")
for nama, tabel in [("jakarta", jakarta), ("bali", bali), ("medan", medan)]:
    print(nama)
    print("  baris  :", len(tabel))
    print("  kolom  :", list(tabel.columns))
    print("  kosong :", tabel.isna().sum().sum())
    print("  waktu  :", tabel.index.min(), "->", tabel.index.max())

ID SENSOR
jakarta
  pm1             17766345.0
  pm25            17766346.0
  relativehumidity 17766347.0
  temperature     17766348.0
  um003           17766349.0
bali
  pm1             17287509.0
  pm25            17287510.0
  relativehumidity 17287511.0
  temperature     17287512.0
  um003           17287513.0
medan
  pm1             14336461.0
  pm25            14336462.0
  um003           14336465.0

RINGKASAN TABEL
jakarta
  baris  : 427
  kolom  : ['pm1', 'pm25', 'um003']
  kosong : 0
  waktu  : 2026-10-01 06:00:00+00:00 -> 2026-10-20 11:00:00+00:00
bali
  baris  : 458
  kolom  : ['pm1', 'pm25', 'um003']
  kosong : 0
  waktu  : 2026-10-01 00:00:00+00:00 -> 2026-10-20 11:00:00+00:00
medan
  baris  : 217
  kolom  : ['pm1', 'pm25', 'um003']
  kosong : 0
  waktu  : 2026-10-11 11:00:00+00:00 -> 2026-10-20 11:00:00+00:00


In [14]:
from pathlib import Path

sensor = {}
for lokasi in daftar_lokasi:
    data = df[df["location_id"] == lokasi].drop_duplicates("parameter")
    sensor[nama_lokasi[lokasi]] = dict(zip(data["parameter"], data["sensor_id"]))

potongan = []
for lokasi in daftar_lokasi:
    nama = nama_lokasi[lokasi]
    tabel = buat_tabel(lokasi).reset_index()
    tabel.insert(0, "lokasi", nama)
    for p in parameter_umum:
        tabel["sensor_" + p] = sensor[nama][p]
    potongan.append(tabel)

gabungan = pd.concat(potongan, ignore_index=True)

gabungan.to_csv(folder + "/gabungan.csv", index=False)
ukuran_kb = Path(folder + "/gabungan.csv").stat().st_size / 1024

print("Jumlah baris dan kolom")
print("  baris :", gabungan.shape[0])
print("  kolom :", gabungan.shape[1])
print("  nama  :", list(gabungan.columns))
print()
print("Baris per lokasi")
print(gabungan["lokasi"].value_counts().to_string())
print()
print("Periode dan cakupan wilayah")
for nama, data in gabungan.groupby("lokasi"):
    print("  {:8} {} -> {}".format(nama, data["waktu"].min(), data["waktu"].max()))
print()
print("Ukuran berkas dan format")
print("  berkas :", folder + "/gabungan.csv")
print("  ukuran :", round(ukuran_kb, 1), "KB")
print("  format : CSV")
print()
print("Tipe variabel penting")
print(gabungan.dtypes.to_string())
print()
print("Nilai kosong per kolom")
print(gabungan.isna().sum().to_string())

Jumlah baris dan kolom
  baris : 1102
  kolom : 8
  nama  : ['lokasi', 'waktu', 'pm1', 'pm25', 'um003', 'sensor_pm1', 'sensor_pm25', 'sensor_um003']

Baris per lokasi
lokasi
bali       458
jakarta    427
medan      217

Periode dan cakupan wilayah
  bali     2026-10-01 00:00:00+00:00 -> 2026-10-20 11:00:00+00:00
  jakarta  2026-10-01 06:00:00+00:00 -> 2026-10-20 11:00:00+00:00
  medan    2026-10-11 11:00:00+00:00 -> 2026-10-20 11:00:00+00:00

Ukuran berkas dan format
  berkas : data/gabungan.csv
  ukuran : 89.5 KB
  format : CSV

Tipe variabel penting
parameter
lokasi                       object
waktu           datetime64[ns, UTC]
pm1                         float64
pm25                        float64
um003                       float64
sensor_pm1                  float64
sensor_pm25                 float64
sensor_um003                float64

Nilai kosong per kolom
parameter
lokasi          0
waktu           0
pm1             0
pm25            0
um003           0
sensor_pm1      0
se